# 🔍 APPRENTISSAGE NON SUPERVISÉ - CLUSTERING

## Objectifs de ce Notebook

Dans ce notebook, nous allons explorer :
1. **Comprendre** l'apprentissage non supervisé et le clustering
2. **Découvrir** les méthodes de clustering hiérarchique
3. **Maîtriser** l'algorithme K-means
4. **Explorer** les métriques de distance et d'évaluation
5. **Appliquer** ces techniques sur des données réelles

---

## 🧠 PARTIE 1: INTRODUCTION AU CLUSTERING

### Qu'est-ce que l'Apprentissage Non Supervisé ?

**Différence fondamentale :**
- **Supervisé** : On a des étiquettes (y) → Prédire une cible
- **Non supervisé** : Pas d'étiquettes → Découvrir des structures cachées

**Exemple concret :**
- **Supervisé** : "Voici des emails étiquetés SPAM/PAS SPAM, apprends à les classifier"
- **Non supervisé** : "Voici des emails sans étiquettes, trouve des groupes similaires"

### Applications du Clustering

🛒 **Segmentation client** : Grouper les clients par comportement d'achat  
🧬 **Bioinformatique** : Classifier les gènes par fonction  
📊 **Analyse de marché** : Identifier des segments de marché  
🎵 **Recommandation** : Grouper les utilisateurs aux goûts similaires  

In [ ]:
# Configuration de l'environnement
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, load_iris
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

# Configuration graphique
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("🚀 Environnement configuré pour l'exploration des données !")

## 📏 PARTIE 2: MÉTRIQUES DE DISTANCE

### Pourquoi les Distances sont Importantes ?

Le clustering repose sur la notion de **similarité** entre points.  
Plus deux points sont **proches**, plus ils sont **similaires**.

### Principales Métriques de Distance

In [ ]:
# Génération de données d'exemple pour illustrer les distances
np.random.seed(42)
point_a = np.array([1, 2])
point_b = np.array([4, 6])

print("Illustration des métriques de distance")
print(f"Point A: {point_a}")
print(f"Point B: {point_b}")
print("-" * 40)

# 1. Distance Euclidienne (L2)
euclidean_dist = np.sqrt(np.sum((point_a - point_b)**2))
print(f"Distance Euclidienne: {euclidean_dist:.3f}")

# 2. Distance de Manhattan (L1)
manhattan_dist = np.sum(np.abs(point_a - point_b))
print(f"Distance de Manhattan: {manhattan_dist:.3f}")

# 3. Distance de Minkowski (généralisation)
def minkowski_distance(p1, p2, p):
    return np.sum(np.abs(p1 - p2)**p)**(1/p)

minkowski_3 = minkowski_distance(point_a, point_b, 3)
print(f"Distance de Minkowski (p=3): {minkowski_3:.3f}")

# Visualisation des distances
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (dist_name, color) in enumerate([('Euclidienne', 'blue'), ('Manhattan', 'red'), ('Minkowski p=3', 'green')]):
    ax = axes[i]
    ax.scatter(*point_a, color=color, s=100, label='Point A')
    ax.scatter(*point_b, color=color, s=100, label='Point B')
    
    if i == 0:  # Euclidienne
        ax.plot([point_a[0], point_b[0]], [point_a[1], point_b[1]], 
                color=color, linestyle='--', alpha=0.7)
    elif i == 1:  # Manhattan
        ax.plot([point_a[0], point_b[0]], [point_a[1], point_a[1]], 
                color=color, linestyle='--', alpha=0.7)
        ax.plot([point_b[0], point_b[0]], [point_a[1], point_b[1]], 
                color=color, linestyle='--', alpha=0.7)
    
    ax.set_title(f'Distance {dist_name}')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 7)

plt.tight_layout()
plt.show()

## 🌳 PARTIE 3: CLUSTERING HIÉRARCHIQUE ASCENDANT

### Principe de Base

Le clustering hiérarchique construit une **hiérarchie** de clusters :
1. **Début** : Chaque point = 1 cluster
2. **Itération** : Fusionner les 2 clusters les plus proches
3. **Fin** : Tous les points dans 1 seul cluster

### Méthodes de Liaison (Linkage)

**Single Linkage** : Distance minimale entre clusters  
**Complete Linkage** : Distance maximale entre clusters  
**Average Linkage** : Distance moyenne entre clusters  
**Ward** : Minimise la variance intra-cluster  

In [ ]:
# Génération de données pour démonstration
X_demo, y_demo = make_blobs(n_samples=50, centers=3, n_features=2, 
                           random_state=42, cluster_std=1.5)

# Calcul de la matrice de distance
distance_matrix = pdist(X_demo, metric='euclidean')

# Clustering hiérarchique avec différentes méthodes
linkage_methods = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, method in enumerate(linkage_methods):
    # Calcul du linkage
    Z = linkage(X_demo, method=method)
    
    # Dendrogramme
    ax = axes[i]
    dendrogram(Z, ax=ax, truncate_mode='level', p=5)
    ax.set_title(f'Dendrogramme - Méthode {method.capitalize()}')
    ax.set_xlabel('Index des échantillons')
    ax.set_ylabel('Distance')

plt.tight_layout()
plt.show()

# Application du clustering hiérarchique
n_clusters = 3
hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels_hierarchical = hierarchical.fit_predict(X_demo)

# Visualisation des résultats
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_demo[:, 0], X_demo[:, 1], c=y_demo, cmap='viridis', alpha=0.7)
plt.title('Données Originales (vraies classes)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X_demo[:, 0], X_demo[:, 1], c=labels_hierarchical, cmap='viridis', alpha=0.7)
plt.title('Clustering Hiérarchique (Ward)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

print(f"Score de Silhouette (Hiérarchique): {silhouette_score(X_demo, labels_hierarchical):.3f}")

## ⭕ PARTIE 4: ALGORITHME K-MEANS

### Principe de Fonctionnement

K-means cherche à partitionner les données en **K clusters** :

1. **Initialisation** : Placer K centroïdes aléatoirement
2. **Attribution** : Assigner chaque point au centroïde le plus proche
3. **Mise à jour** : Recalculer les centroïdes (moyenne des points)
4. **Répéter** 2-3 jusqu'à convergence

### Avantages et Inconvénients

**✅ Avantages :**
- Simple et rapide
- Fonctionne bien sur des clusters sphériques
- Peu de paramètres

**❌ Inconvénients :**
- Nécessite de choisir K à l'avance
- Sensible à l'initialisation
- Assume des clusters de forme sphérique

In [ ]:
# Démonstration de l'algorithme K-means étape par étape
def plot_kmeans_steps(X, n_clusters=3, max_iter=10):
    """Visualise les étapes de l'algorithme K-means"""
    
    # Initialisation aléatoire des centroïdes
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], n_clusters, replace=False)]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()
    
    for iteration in range(min(6, max_iter)):
        ax = axes[iteration]
        
        # Calcul des distances et attribution des clusters
        distances = np.sqrt(((X - centroids[:, np.newaxis])**2).sum(axis=2))
        labels = np.argmin(distances, axis=0)
        
        # Visualisation
        colors = ['red', 'blue', 'green', 'orange', 'purple']
        for k in range(n_clusters):
            cluster_points = X[labels == k]
            ax.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                      c=colors[k], alpha=0.6, label=f'Cluster {k+1}')
            ax.scatter(centroids[k, 0], centroids[k, 1], 
                      c='black', marker='x', s=200, linewidths=3)
        
        ax.set_title(f'Itération {iteration + 1}')
        ax.set_xlabel('Feature 1')
        ax.set_ylabel('Feature 2')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Mise à jour des centroïdes
        new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(n_clusters)])
        
        # Vérification de la convergence
        if np.allclose(centroids, new_centroids, rtol=1e-4):
            print(f"Convergence atteinte à l'itération {iteration + 1}")
            break
            
        centroids = new_centroids
    
    plt.tight_layout()
    plt.show()
    
    return labels, centroids

# Application sur nos données
print("Visualisation des étapes de K-means:")
final_labels, final_centroids = plot_kmeans_steps(X_demo, n_clusters=3)

## 📊 PARTIE 5: MÉTRIQUES D'ÉVALUATION

### Comment Évaluer un Clustering ?

**Défi** : Pas de "vraie réponse" en non supervisé !

### Métriques Internes (sans vérité terrain)

**1. Inertie (Within-Cluster Sum of Squares)**
- Somme des distances au carré des points à leur centroïde
- Plus faible = meilleur

**2. Score de Silhouette**
- Mesure la cohésion interne vs séparation externe
- Entre -1 et 1, plus élevé = meilleur

In [ ]:
# Évaluation avec différentes métriques
def evaluate_clustering(X, labels, centroids=None):
    """Évalue la qualité d'un clustering"""
    
    # Score de Silhouette
    silhouette = silhouette_score(X, labels)
    
    # Inertie (si centroids disponibles)
    if centroids is not None:
        inertia = sum([np.sum((X[labels == k] - centroids[k])**2) 
                      for k in range(len(centroids))])
    else:
        # Calcul approximatif de l'inertie
        inertia = 0
        for k in np.unique(labels):
            cluster_points = X[labels == k]
            centroid = cluster_points.mean(axis=0)
            inertia += np.sum((cluster_points - centroid)**2)
    
    return silhouette, inertia

# Comparaison K-means vs Hiérarchique
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_demo)

sil_kmeans, inertia_kmeans = evaluate_clustering(X_demo, labels_kmeans, kmeans.cluster_centers_)
sil_hierarchical, inertia_hierarchical = evaluate_clustering(X_demo, labels_hierarchical)

print("Comparaison des algorithmes:")
print("-" * 40)
print(f"K-means:")
print(f"  Silhouette Score: {sil_kmeans:.3f}")
print(f"  Inertie: {inertia_kmeans:.3f}")
print(f"\nHiérarchique:")
print(f"  Silhouette Score: {sil_hierarchical:.3f}")
print(f"  Inertie: {inertia_hierarchical:.3f}")

# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Données originales
axes[0].scatter(X_demo[:, 0], X_demo[:, 1], c=y_demo, cmap='viridis', alpha=0.7)
axes[0].set_title('Données Originales')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# K-means
axes[1].scatter(X_demo[:, 0], X_demo[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.7)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                c='red', marker='x', s=200, linewidths=3)
axes[1].set_title(f'K-means (Silhouette: {sil_kmeans:.3f})')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

# Hiérarchique
axes[2].scatter(X_demo[:, 0], X_demo[:, 1], c=labels_hierarchical, cmap='viridis', alpha=0.7)
axes[2].set_title(f'Hiérarchique (Silhouette: {sil_hierarchical:.3f})')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 🔍 PARTIE 6: MÉTHODE DU COUDE (ELBOW METHOD)

### Comment Choisir le Nombre de Clusters ?

La **méthode du coude** aide à déterminer le nombre optimal de clusters :
1. Tester différentes valeurs de K
2. Calculer l'inertie pour chaque K
3. Chercher le "coude" dans la courbe

In [ ]:
# Méthode du coude pour déterminer le nombre optimal de clusters
def elbow_method(X, max_k=10):
    """Applique la méthode du coude"""
    
    inertias = []
    silhouette_scores = []
    k_range = range(2, max_k + 1)
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X, labels))
    
    return k_range, inertias, silhouette_scores

# Application de la méthode du coude
k_range, inertias, silhouette_scores = elbow_method(X_demo, max_k=8)

# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Courbe de l'inertie
ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Nombre de clusters (K)')
ax1.set_ylabel('Inertie')
ax1.set_title('Méthode du Coude - Inertie')
ax1.grid(True, alpha=0.3)

# Annotation du coude potentiel
optimal_k = 3  # Basé sur l'observation visuelle
ax1.annotate(f'Coude potentiel\nK = {optimal_k}', 
             xy=(optimal_k, inertias[optimal_k-2]), 
             xytext=(optimal_k+1, inertias[optimal_k-2]+50),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=12, color='red')

# Score de Silhouette
ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Nombre de clusters (K)')
ax2.set_ylabel('Score de Silhouette')
ax2.set_title('Score de Silhouette par K')
ax2.grid(True, alpha=0.3)

# Annotation du maximum
max_sil_idx = np.argmax(silhouette_scores)
max_sil_k = k_range[max_sil_idx]
ax2.annotate(f'Maximum\nK = {max_sil_k}', 
             xy=(max_sil_k, silhouette_scores[max_sil_idx]), 
             xytext=(max_sil_k+1, silhouette_scores[max_sil_idx]-0.05),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=12, color='red')

plt.tight_layout()
plt.show()

print("Résultats de l'analyse:")
print(f"K optimal selon l'inertie (méthode du coude): {optimal_k}")
print(f"K optimal selon le score de Silhouette: {max_sil_k}")